# ========================================
# VENDOR ANALYSIS PROJECT - VISUALIZATIONS & DASHBOARDS
# Notebook 5: Interactive Charts & Business Intelligence Dashboards
# ========================================

In [1]:
# CELL 1: Setup
from google.colab import drive
drive.mount('/content/drive')

import sys
sys.path.append('/content/drive/MyDrive/vendor-analysis-project')

import pandas as pd
import numpy as np
import plotly.graph_objects as go
import plotly.express as px
from plotly.subplots import make_subplots
import matplotlib.pyplot as plt
import seaborn as sns
import os
import glob
from datetime import datetime

print("✅ Setup complete\n")


Mounted at /content/drive
✅ Setup complete



In [2]:
# CELL 2: Load Enriched Data from Exports
EXPORT_DIR = '/content/drive/MyDrive/vendor-analysis-project/data/exports'

# Load main enriched dataset
enriched_files = glob.glob(f'{EXPORT_DIR}/vendors_enriched.csv')

if not enriched_files:
    print("❌ No enriched data found!")
else:
    enriched_file = enriched_files[0]
    print(f"📁 Loading enriched data: {os.path.basename(enriched_file)}\n")

    df = pd.read_csv(enriched_file, low_memory=False)
    print(f"✅ Loaded {len(df):,} vendors")
    print(f"📊 Columns: {len(df.columns)}\n")


📁 Loading enriched data: vendors_enriched.csv

✅ Loaded 100 vendors
📊 Columns: 59



In [3]:
# CELL 3: Create Output Directory for Visualizations
VIZ_DIR = '/content/drive/MyDrive/vendor-analysis-project/data/exports/visualizations'
os.makedirs(VIZ_DIR, exist_ok=True)

print(f"📁 Visualizations will be saved to: {VIZ_DIR}\n")


📁 Visualizations will be saved to: /content/drive/MyDrive/vendor-analysis-project/data/exports/visualizations



In [4]:
# CELL 4: Visualization 1 - Vendor Tier Distribution (Pie Chart)
print("Creating Visualization 1: Vendor Tier Distribution...\n")

tier_counts = df['vendor_tier'].value_counts()
colors_tier = ['#2ecc71', '#3498db', '#f39c12', '#e74c3c', '#95a5a6']

fig_tier = go.Figure(data=[go.Pie(
    labels=tier_counts.index,
    values=tier_counts.values,
    marker=dict(colors=colors_tier),
    textinfo='label+percent+value',
    textposition='auto',
    hovertemplate='<b>%{label}</b><br>Count: %{value}<br>Percentage: %{percent}<extra></extra>'
)])

fig_tier.update_layout(
    title={
        'text': '<b>Vendor Tier Distribution</b><br><sub>Segmentation by Annual Spend</sub>',
        'x': 0.5,
        'xanchor': 'center',
        'font': {'size': 20}
    },
    height=600,
    showlegend=True,
    hovermode='closest',
    font=dict(family="Arial, sans-serif", size=12),
    margin=dict(l=50, r=50, t=100, b=50)
)

fig_tier.write_html(f'{VIZ_DIR}/01_vendor_tier_distribution.html')
print("✓ Saved: 01_vendor_tier_distribution.html\n")


Creating Visualization 1: Vendor Tier Distribution...

✓ Saved: 01_vendor_tier_distribution.html



In [5]:
# CELL 5: Visualization 2 - Total Spend by State (Top 15)
print("Creating Visualization 2: Top 15 States by Spend...\n")

state_spend = df[df['state'] != ''].groupby('state')['total_spend'].sum().nlargest(15).sort_values()

fig_state = go.Figure(data=[
    go.Bar(
        x=state_spend.values,
        y=state_spend.index,
        orientation='h',
        marker=dict(
            color=state_spend.values,
            colorscale='Viridis',
            showscale=True,
            colorbar=dict(title="Total Spend ($)")
        ),
        text=[f'${v/1e6:.2f}M' for v in state_spend.values],
        textposition='auto',
        hovertemplate='<b>%{y}</b><br>Total Spend: $%{x:,.0f}<extra></extra>'
    )
])

fig_state.update_layout(
    title={
        'text': '<b>Top 15 States by Total Vendor Spend</b>',
        'x': 0.5,
        'xanchor': 'center',
        'font': {'size': 20}
    },
    xaxis_title='Total Spend ($)',
    yaxis_title='State',
    height=600,
    hovermode='closest',
    margin=dict(l=100, r=50, t=100, b=50)
)

fig_state.write_html(f'{VIZ_DIR}/02_top_states_by_spend.html')
print("✓ Saved: 02_top_states_by_spend.html\n")


Creating Visualization 2: Top 15 States by Spend...

✓ Saved: 02_top_states_by_spend.html



In [6]:
# CELL 6: Visualization 3 - Vendor Health Status Distribution
print("Creating Visualization 3: Vendor Health Status...\n")

health_counts = df['vendor_health_status'].value_counts()
health_colors = {
    'Healthy': '#2ecc71',
    'At Risk': '#f39c12',
    'Critical': '#e74c3c',
    'Inactive': '#95a5a6',
    'Unknown': '#34495e'
}
health_chart_colors = [health_colors.get(h, '#95a5a6') for h in health_counts.index]

fig_health = go.Figure(data=[go.Bar(
    x=health_counts.index,
    y=health_counts.values,
    marker=dict(color=health_chart_colors),
    text=health_counts.values,
    textposition='auto',
    hovertemplate='<b>%{x}</b><br>Vendors: %{y}<extra></extra>'
)])

fig_health.update_layout(
    title={
        'text': '<b>Vendor Health Status Distribution</b><br><sub>Based on Spend Trends & Activity</sub>',
        'x': 0.5,
        'xanchor': 'center',
        'font': {'size': 20}
    },
    xaxis_title='Health Status',
    yaxis_title='Number of Vendors',
    height=500,
    hovermode='x unified',
    margin=dict(l=50, r=50, t=100, b=50)
)

fig_health.write_html(f'{VIZ_DIR}/03_vendor_health_status.html')
print("✓ Saved: 03_vendor_health_status.html\n")


Creating Visualization 3: Vendor Health Status...

✓ Saved: 03_vendor_health_status.html



In [7]:
# CELL 7: Visualization 4 - Spend Trend (2022-2024)
print("Creating Visualization 4: Year-over-Year Spend Trend...\n")

years = ['2022', '2023', '2024']
total_spend_by_year = [
    df['spend_2022'].sum(),
    df['spend_2023'].sum(),
    df['spend_2024'].sum()
]

fig_trend = go.Figure()

fig_trend.add_trace(go.Scatter(
    x=years,
    y=total_spend_by_year,
    mode='lines+markers',
    name='Total Spend',
    line=dict(color='#3498db', width=4),
    marker=dict(size=12, symbol='circle'),
    fill='tozeroy',
    hovertemplate='<b>%{x}</b><br>Total Spend: $%{y:,.0f}<extra></extra>'
))

# Add YoY growth rates as annotations
for i in range(1, len(years)):
    growth = ((total_spend_by_year[i] - total_spend_by_year[i-1]) / total_spend_by_year[i-1] * 100)
    fig_trend.add_annotation(
        x=years[i],
        y=total_spend_by_year[i],
        text=f'{growth:+.1f}%',
        showarrow=True,
        arrowhead=2,
        arrowsize=1,
        arrowwidth=2,
        arrowcolor='#e74c3c' if growth < 0 else '#2ecc71',
        ax=0,
        ay=-30,
        font=dict(size=12, color='#e74c3c' if growth < 0 else '#2ecc71')
    )

fig_trend.update_layout(
    title={
        'text': '<b>Vendor Portfolio Spend Trend (2022-2024)</b><br><sub>Year-over-Year Analysis</sub>',
        'x': 0.5,
        'xanchor': 'center',
        'font': {'size': 20}
    },
    xaxis_title='Year',
    yaxis_title='Total Spend ($)',
    height=500,
    hovermode='x',
    margin=dict(l=70, r=50, t=100, b=50),
    yaxis=dict(tickformat='$,.0f')
)

fig_trend.write_html(f'{VIZ_DIR}/04_spend_trend_2022_2024.html')
print("✓ Saved: 04_spend_trend_2022_2024.html\n")


Creating Visualization 4: Year-over-Year Spend Trend...

✓ Saved: 04_spend_trend_2022_2024.html



In [8]:
# CELL 8: Visualization 5 - Top 15 Vendors by Spend
print("Creating Visualization 5: Top 15 Vendors by Spend...\n")

top_vendors = df.nlargest(15, 'total_spend')[['vendor_name', 'total_spend', 'state']]

fig_top = go.Figure(data=[
    go.Bar(
        y=top_vendors['vendor_name'].values,
        x=top_vendors['total_spend'].values,
        orientation='h',
        marker=dict(
            color=top_vendors['total_spend'].values,
            colorscale='Blues',
            showscale=True,
            colorbar=dict(title="Spend ($)")
        ),
        text=[f'${v/1e6:.2f}M' for v in top_vendors['total_spend'].values],
        textposition='auto',
        customdata=top_vendors['state'].values,
        hovertemplate='<b>%{y}</b><br>State: %{customdata}<br>Spend: $%{x:,.0f}<extra></extra>'
    )
])

fig_top.update_layout(
    title={
        'text': '<b>Top 15 Vendors by Total Spend</b>',
        'x': 0.5,
        'xanchor': 'center',
        'font': {'size': 20}
    },
    xaxis_title='Total Spend ($)',
    yaxis_title='Vendor Name',
    height=700,
    hovermode='closest',
    margin=dict(l=300, r=50, t=100, b=50)
)

fig_top.write_html(f'{VIZ_DIR}/05_top_15_vendors.html')
print("✓ Saved: 05_top_15_vendors.html\n")


Creating Visualization 5: Top 15 Vendors by Spend...

✓ Saved: 05_top_15_vendors.html



In [9]:
# CELL 9: Visualization 6 - Spend Growth Rate Distribution
print("Creating Visualization 6: Spend Growth Rate Distribution...\n")

growth_df = df[df['spend_2023'] > 0].copy()

fig_growth = go.Figure(data=[
    go.Histogram(
        x=growth_df['spend_growth_2023_2024'],
        nbinsx=50,
        marker=dict(
            color=growth_df['spend_growth_2023_2024'],
            colorscale='RdYlGn',
            showscale=True,
            colorbar=dict(title="Growth Rate (%)")
        ),
        hovertemplate='Growth Rate: %{x:.1f}%<br>Vendors: %{y}<extra></extra>'
    )
])

fig_growth.update_layout(
    title={
        'text': '<b>Spend Growth Rate Distribution (2023-2024)</b><br><sub>For Vendors with 2023 Spend</sub>',
        'x': 0.5,
        'xanchor': 'center',
        'font': {'size': 20}
    },
    xaxis_title='Growth Rate (%)',
    yaxis_title='Number of Vendors',
    height=500,
    hovermode='x',
    margin=dict(l=50, r=50, t=100, b=50),
    xaxis=dict(zeroline=True, zerolinewidth=2, zerolinecolor='red')
)

fig_growth.write_html(f'{VIZ_DIR}/06_growth_rate_distribution.html')
print("✓ Saved: 06_growth_rate_distribution.html\n")


Creating Visualization 6: Spend Growth Rate Distribution...

✓ Saved: 06_growth_rate_distribution.html



In [10]:
# CELL 10: Visualization 7 - Performance Score Distribution
print("Creating Visualization 7: Performance Score Distribution...\n")

fig_perf = go.Figure(data=[
    go.Histogram(
        x=df['performance_score'],
        nbinsx=30,
        marker=dict(color='#3498db'),
        hovertemplate='Performance Score: %{x:.0f}<br>Vendors: %{y}<extra></extra>'
    )
])

fig_perf.update_layout(
    title={
        'text': '<b>Vendor Performance Score Distribution</b><br><sub>0-100 Scale: Spend, Growth, Activity</sub>',
        'x': 0.5,
        'xanchor': 'center',
        'font': {'size': 20}
    },
    xaxis_title='Performance Score (0-100)',
    yaxis_title='Number of Vendors',
    height=500,
    hovermode='x',
    margin=dict(l=50, r=50, t=100, b=50)
)

fig_perf.write_html(f'{VIZ_DIR}/07_performance_score_distribution.html')
print("✓ Saved: 07_performance_score_distribution.html\n")


Creating Visualization 7: Performance Score Distribution...

✓ Saved: 07_performance_score_distribution.html



In [11]:
# CELL 11: Visualization 8 - Vendor Type Analysis
print("Creating Visualization 8: Top Vendor Types by Spend...\n")

vendor_type_spend = df[df['vendor_type'] != ''].groupby('vendor_type')['total_spend'].agg(['sum', 'count']).nlargest(10, 'sum')

fig_type = go.Figure(data=[
    go.Bar(
        x=vendor_type_spend.index,
        y=vendor_type_spend['sum'],
        text=[f'${v/1e6:.1f}M<br>({c} vendors)' for v, c in zip(vendor_type_spend['sum'], vendor_type_spend['count'])],
        textposition='auto',
        marker=dict(color=vendor_type_spend['sum'], colorscale='Plasma', showscale=True),
        hovertemplate='<b>%{x}</b><br>Total Spend: $%{y:,.0f}<extra></extra>'
    )
])

fig_type.update_layout(
    title={
        'text': '<b>Top 10 Vendor Types by Total Spend</b>',
        'x': 0.5,
        'xanchor': 'center',
        'font': {'size': 20}
    },
    xaxis_title='Vendor Type',
    yaxis_title='Total Spend ($)',
    height=600,
    hovermode='x unified',
    margin=dict(l=50, r=50, t=100, b=100)
)

fig_type.update_xaxes(tickangle=-45)

fig_type.write_html(f'{VIZ_DIR}/08_vendor_type_analysis.html')
print("✓ Saved: 08_vendor_type_analysis.html\n")

Creating Visualization 8: Top Vendor Types by Spend...

✓ Saved: 08_vendor_type_analysis.html



In [12]:
# CELL 12: Visualization 9 - Engagement Level Distribution
print("Creating Visualization 9: Engagement Level Distribution...\n")

engagement_counts = df['engagement_level'].value_counts()
engagement_colors = {
    'Highly Engaged': '#2ecc71',
    'Moderately Engaged': '#3498db',
    'Low Engagement': '#f39c12',
    'No Engagement': '#e74c3c'
}
engagement_chart_colors = [engagement_colors.get(e, '#95a5a6') for e in engagement_counts.index]

fig_engagement = go.Figure(data=[go.Pie(
    labels=engagement_counts.index,
    values=engagement_counts.values,
    marker=dict(colors=engagement_chart_colors),
    textinfo='label+percent+value',
    textposition='auto',
    hovertemplate='<b>%{label}</b><br>Vendors: %{value}<br>Percentage: %{percent}<extra></extra>'
)])

fig_engagement.update_layout(
    title={
        'text': '<b>Vendor Engagement Level Distribution</b><br><sub>Based on Activity Score</sub>',
        'x': 0.5,
        'xanchor': 'center',
        'font': {'size': 20}
    },
    height=600,
    hovermode='closest',
    margin=dict(l=50, r=50, t=100, b=50)
)

fig_engagement.write_html(f'{VIZ_DIR}/09_engagement_distribution.html')
print("✓ Saved: 09_engagement_distribution.html\n")


Creating Visualization 9: Engagement Level Distribution...

✓ Saved: 09_engagement_distribution.html



In [13]:
# CELL 13: Visualization 10 - Risk Category Analysis
print("Creating Visualization 10: Risk Category Analysis...\n")

risk_counts = df['risk_category'].value_counts()
risk_colors_map = {
    'No Activity': '#95a5a6',
    'Growing/Stable': '#2ecc71',
    'Low Risk': '#3498db',
    'Medium Risk': '#f39c12',
    'High Risk': '#e74c3c'
}
risk_chart_colors = [risk_colors_map.get(r, '#95a5a6') for r in risk_counts.index]

fig_risk = go.Figure(data=[go.Bar(
    x=risk_counts.index,
    y=risk_counts.values,
    marker=dict(color=risk_chart_colors),
    text=risk_counts.values,
    textposition='auto',
    hovertemplate='<b>%{x}</b><br>Vendors: %{y}<extra></extra>'
)])

fig_risk.update_layout(
    title={
        'text': '<b>Vendor Risk Category Distribution</b><br><sub>Based on Spend Growth Trends</sub>',
        'x': 0.5,
        'xanchor': 'center',
        'font': {'size': 20}
    },
    xaxis_title='Risk Category',
    yaxis_title='Number of Vendors',
    height=500,
    hovermode='x unified',
    margin=dict(l=50, r=50, t=100, b=50)
)

fig_risk.write_html(f'{VIZ_DIR}/10_risk_category_distribution.html')
print("✓ Saved: 10_risk_category_distribution.html\n")

Creating Visualization 10: Risk Category Analysis...

✓ Saved: 10_risk_category_distribution.html



In [14]:
# CELL 14: Visualization 11 - Scatter: Performance vs Spend
print("Creating Visualization 11: Performance Score vs Total Spend...\n")

fig_scatter = px.scatter(
    df[df['total_spend'] > 0],
    x='total_spend',
    y='performance_score',
    size='activity_score',
    color='vendor_health_status',
    hover_name='vendor_name',
    hover_data={'state': True, 'total_spend': ':.0f', 'performance_score': ':.0f'},
    title='Performance Score vs Total Spend',
    labels={
        'total_spend': 'Total Spend ($)',
        'performance_score': 'Performance Score (0-100)',
        'vendor_health_status': 'Health Status'
    },
    color_discrete_map={
        'Healthy': '#2ecc71',
        'At Risk': '#f39c12',
        'Critical': '#e74c3c',
        'Inactive': '#95a5a6',
        'Unknown': '#34495e'
    }
)

fig_scatter.update_layout(
    height=700,
    xaxis_type='log',
    hovermode='closest',
    margin=dict(l=50, r=50, t=100, b=50)
)

fig_scatter.update_xaxes(tickformat='$,.0f')

fig_scatter.write_html(f'{VIZ_DIR}/11_performance_vs_spend_scatter.html')
print("✓ Saved: 11_performance_vs_spend_scatter.html\n")


Creating Visualization 11: Performance Score vs Total Spend...

✓ Saved: 11_performance_vs_spend_scatter.html



In [15]:
# CELL 15: Visualization 12 - Contact Quality Analysis
print("Creating Visualization 12: Contact Quality Analysis...\n")

contact_quality_dist = df['contact_quality_score'].value_counts().sort_index()
contact_labels = {0: 'No Contact Info', 50: 'Email Only / Phone Only', 100: 'Email & Phone'}
contact_labels_mapped = [contact_labels.get(int(q), str(q)) for q in contact_quality_dist.index]

fig_contact = go.Figure(data=[go.Bar(
    x=contact_labels_mapped,
    y=contact_quality_dist.values,
    marker=dict(color=['#e74c3c', '#f39c12', '#2ecc71']),
    text=contact_quality_dist.values,
    textposition='auto',
    hovertemplate='<b>%{x}</b><br>Vendors: %{y}<extra></extra>'
)])

fig_contact.update_layout(
    title={
        'text': '<b>Vendor Contact Quality Distribution</b><br><sub>Email & Phone Availability</sub>',
        'x': 0.5,
        'xanchor': 'center',
        'font': {'size': 20}
    },
    xaxis_title='Contact Quality',
    yaxis_title='Number of Vendors',
    height=500,
    hovermode='x unified',
    margin=dict(l=50, r=50, t=100, b=50)
)

fig_contact.write_html(f'{VIZ_DIR}/12_contact_quality_analysis.html')
print("✓ Saved: 12_contact_quality_analysis.html\n")


Creating Visualization 12: Contact Quality Analysis...

✓ Saved: 12_contact_quality_analysis.html



In [16]:
# CELL 16: Create Summary Dashboard HTML
print("Creating Summary Dashboard...\n")

dashboard_html = """
<!DOCTYPE html>
<html>
<head>
    <meta charset="utf-8">
    <title>Vendor Analysis Dashboard</title>
    <script src="https://cdn.plot.ly/plotly-latest.min.js"></script>
    <style>
        * {
            margin: 0;
            padding: 0;
            box-sizing: border-box;
        }

        body {
            font-family: 'Segoe UI', Tahoma, Geneva, Verdana, sans-serif;
            background: linear-gradient(135deg, #667eea 0%, #764ba2 100%);
            color: #333;
            padding: 20px;
        }

        .container {
            max-width: 1400px;
            margin: 0 auto;
        }

        header {
            background: white;
            padding: 30px;
            border-radius: 10px;
            margin-bottom: 30px;
            box-shadow: 0 2px 10px rgba(0,0,0,0.1);
        }

        h1 {
            color: #667eea;
            margin-bottom: 10px;
            font-size: 32px;
        }

        .subtitle {
            color: #7f8c8d;
            font-size: 16px;
        }

        .dashboard {
            display: grid;
            grid-template-columns: repeat(auto-fit, minmax(500px, 1fr));
            gap: 20px;
            margin-bottom: 30px;
        }

        .card {
            background: white;
            border-radius: 10px;
            box-shadow: 0 2px 10px rgba(0,0,0,0.1);
            overflow: hidden;
            transition: transform 0.3s ease, box-shadow 0.3s ease;
        }

        .card:hover {
            transform: translateY(-5px);
            box-shadow: 0 5px 20px rgba(0,0,0,0.15);
        }

        .card-title {
            background: linear-gradient(135deg, #667eea 0%, #764ba2 100%);
            color: white;
            padding: 15px;
            font-weight: bold;
            font-size: 14px;
        }

        .card-content {
            padding: 20px;
        }

        .stat-box {
            display: flex;
            flex-direction: column;
            align-items: center;
            padding: 20px;
            background: #f8f9fa;
            border-radius: 8px;
            margin-bottom: 15px;
        }

        .stat-value {
            font-size: 32px;
            font-weight: bold;
            color: #667eea;
        }

        .stat-label {
            font-size: 12px;
            color: #7f8c8d;
            margin-top: 5px;
            text-align: center;
        }

        .chart-link {
            display: inline-block;
            padding: 12px 20px;
            background: linear-gradient(135deg, #667eea 0%, #764ba2 100%);
            color: white;
            text-decoration: none;
            border-radius: 5px;
            margin: 5px;
            transition: opacity 0.3s ease;
            font-size: 12px;
        }

        .chart-link:hover {
            opacity: 0.8;
        }

        .links-section {
            display: flex;
            flex-wrap: wrap;
            gap: 10px;
            margin-top: 15px;
        }

        footer {
            background: white;
            padding: 20px;
            border-radius: 10px;
            text-align: center;
            color: #7f8c8d;
            font-size: 12px;
            margin-top: 30px;
        }
    </style>
</head>
<body>
    <div class="container">
        <header>
            <h1>Vendor Analysis Dashboard</h1>
            <p class="subtitle">Comprehensive Business Intelligence & Performance Metrics</p>
        </header>

        <div class="dashboard">
            <div class="card">
                <div class="card-title">Portfolio Overview</div>
                <div class="card-content">
                    <div class="stat-box">
                        <div class="stat-value">""" + str(len(df)) + """</div>
                        <div class="stat-label">Total Vendors</div>
                    </div>
                    <div class="stat-box">
                        <div class="stat-value">$""" + str(int(df['total_spend'].sum()/1e6)) + """M</div>
                        <div class="stat-label">Total Spend (All Years)</div>
                    </div>
                    <div class="stat-box">
                        <div class="stat-value">$""" + str(int(df[df['total_spend']>0]['total_spend'].mean()/1e3)) + """K</div>
                        <div class="stat-label">Average Spend per Vendor</div>
                    </div>
                </div>
            </div>

            <div class="card">
                <div class="card-title">Spend Trends</div>
                <div class="card-content">
                    <div class="stat-box">
                        <div class="stat-value">$""" + str(int(df['spend_2024'].sum()/1e6)) + """M</div>
                        <div class="stat-label">2024 YTD Spend</div>
                    </div>
                    <div class="stat-box">
                        <div class="stat-value">$""" + str(int(df['spend_2023'].sum()/1e6)) + """M</div>
                        <div class="stat-label">2023 Total Spend</div>
                    </div>
                    <div class="stat-box">
                        <div class="stat-value">""" + str(int((df['spend_2024'].sum() - df['spend_2023'].sum())/df['spend_2023'].sum()*100)) + """%</div>
                        <div class="stat-label">YoY Growth Rate</div>
                    </div>
                </div>
            </div>

            <div class="card">
                <div class="card-title">Vendor Health</div>
                <div class="card-content">
                    <div class="stat-box">
                        <div class="stat-value">""" + str(len(df[df['vendor_health_status']=='Healthy'])) + """</div>
                        <div class="stat-label">Healthy Vendors</div>
                    </div>
                    <div class="stat-box">
                        <div class="stat-value">""" + str(len(df[df['vendor_health_status']=='At Risk'])) + """</div>
                        <div class="stat-label">At Risk Vendors</div>
                    </div>
                    <div class="stat-box">
                        <div class="stat-value">""" + str(len(df[df['vendor_health_status']=='Critical'])) + """</div>
                        <div class="stat-label">Critical Vendors</div>
                    </div>
                </div>
            </div>

            <div class="card">
                <div class="card-title">Geographic Reach</div>
                <div class="card-content">
                    <div class="stat-box">
                        <div class="stat-value">""" + str(df[df['state']!='']['state'].nunique()) + """</div>
                        <div class="stat-label">States Covered</div>
                    </div>
                    <div class="stat-box">
                        <div class="stat-value">""" + str(len(df[df['primary_email']!=''])) + """</div>
                        <div class="stat-label">Vendors with Email</div>
                    </div>
                    <div class="stat-box">
                        <div class="stat-value">""" + str(len(df[df['phone']!=''])) + """</div>
                        <div class="stat-label">Vendors with Phone</div>
                    </div>
                </div>
            </div>
        </div>

        <div class="card">
            <div class="card-title">Available Visualizations</div>
            <div class="card-content">
                <div class="links-section">
                    <a href="01_vendor_tier_distribution.html" class="chart-link">Tier Distribution</a>
                    <a href="02_top_states_by_spend.html" class="chart-link">Top States</a>
                    <a href="03_vendor_health_status.html" class="chart-link">Health Status</a>
                    <a href="04_spend_trend_2022_2024.html" class="chart-link">Spend Trends</a>
                    <a href="05_top_15_vendors.html" class="chart-link">Top Vendors</a>
                    <a href="06_growth_rate_distribution.html" class="chart-link">Growth Rates</a>
                    <a href="07_performance_score_distribution.html" class="chart-link">Performance Scores</a>
                    <a href="08_vendor_type_analysis.html" class="chart-link">Vendor Types</a>
                    <a href="09_engagement_distribution.html" class="chart-link">Engagement Levels</a>
                    <a href="10_risk_category_distribution.html" class="chart-link">Risk Categories</a>
                    <a href="11_performance_vs_spend_scatter.html" class="chart-link">Performance vs Spend</a>
                    <a href="12_contact_quality_analysis.html" class="chart-link">Contact Quality</a>
                </div>
            </div>
        </div>

        <footer>
            <p>Vendor Analysis Dashboard | Generated """ + datetime.now().strftime("%Y-%m-%d %H:%M:%S") + """</p>
            <p>All data is based on the latest vendor master extract and processed through PySpark analytics pipeline</p>
        </footer>
    </div>
</body>
</html>
"""

with open(f'{VIZ_DIR}/index.html', 'w') as f:
    f.write(dashboard_html)

print("✓ Saved: index.html (Dashboard Home)\n")


Creating Summary Dashboard...

✓ Saved: index.html (Dashboard Home)



In [17]:
# CELL 17: Export Summary Statistics
print("Creating Summary Statistics Report...\n")

summary_stats = {
    'Total Vendors': len(df),
    'Total Spend (All Years)': f"${df['total_spend'].sum():,.0f}",
    'Average Spend per Vendor': f"${df['total_spend'].mean():,.0f}",
    'Median Spend': f"${df['total_spend'].median():,.0f}",
    'Max Vendor Spend': f"${df['total_spend'].max():,.0f}",
    'Min Vendor Spend (>0)': f"${df[df['total_spend']>0]['total_spend'].min():,.0f}",
    '2024 YTD Spend': f"${df['spend_2024'].sum():,.0f}",
    '2023 Total Spend': f"${df['spend_2023'].sum():,.0f}",
    'YoY Growth Rate': f"{((df['spend_2024'].sum() - df['spend_2023'].sum())/df['spend_2023'].sum()*100):.2f}%",
    'Vendors with Email': len(df[df['primary_email']!='']) ,
    'Vendors with Phone': len(df[df['phone']!='']),
    'States Covered': df[df['state']!='']['state'].nunique(),
    'Healthy Vendors': len(df[df['vendor_health_status']=='Healthy']),
    'At Risk Vendors': len(df[df['vendor_health_status']=='At Risk']),
    'Critical Vendors': len(df[df['vendor_health_status']=='Critical']),
    'Inactive Vendors': len(df[df['vendor_health_status']=='Inactive']),
    'Avg Performance Score': f"{df['performance_score'].mean():.1f}/100",
    'Avg Activity Score': f"{df['activity_score'].mean():.1f}/100",
}

summary_df = pd.DataFrame(list(summary_stats.items()), columns=['Metric', 'Value'])
summary_df.to_csv(f'{VIZ_DIR}/summary_statistics.csv', index=False)

print("Summary Statistics:")
print(summary_df.to_string(index=False))

print(f"\n✓ Saved: summary_statistics.csv\n")

# CELL 18: Final Summary
print("="*100)
print("VISUALIZATION COMPLETE!")
print("="*100)

print(f"\n📊 Files Created in: {VIZ_DIR}")
print("\n12 Interactive Visualizations:")
print("  1. Vendor Tier Distribution")
print("  2. Top 15 States by Spend")
print("  3. Vendor Health Status")
print("  4. Spend Trend 2022-2024")
print("  5. Top 15 Vendors")
print("  6. Growth Rate Distribution")
print("  7. Performance Score Distribution")
print("  8. Vendor Type Analysis")
print("  9. Engagement Distribution")
print("  10. Risk Category Distribution")
print("  11. Performance vs Spend Scatter")
print("  12. Contact Quality Analysis")
print("\nPlus:")
print("  - Dashboard Index (index.html)")
print("  - Summary Statistics (summary_statistics.csv)")

print("\n" + "="*100)
print("✅ ALL ANALYSIS COMPLETE!")
print("="*100)
print("\nProject Timeline:")
print("  ✅ Step 1: Environment Setup")
print("  ✅ Step 2: Web Scraping (Selenium)")
print("  ✅ Step 3: Database Loading")
print("  ✅ Step 4: PySpark Big Data Processing")
print("  ✅ Step 5: Interactive Visualizations")
print("  ➡️  Step 6: Executive Report (PDF/PPT)")
print("\n" + "="*100)

Creating Summary Statistics Report...

Summary Statistics:
                  Metric       Value
           Total Vendors         100
 Total Spend (All Years) $66,043,915
Average Spend per Vendor    $660,439
            Median Spend          $0
        Max Vendor Spend $23,752,511
   Min Vendor Spend (>0)        $245
          2024 YTD Spend $24,706,696
        2023 Total Spend $13,462,182
         YoY Growth Rate      83.53%
      Vendors with Email         100
      Vendors with Phone         100
          States Covered          17
         Healthy Vendors          17
         At Risk Vendors           2
        Critical Vendors          26
        Inactive Vendors          55
   Avg Performance Score    49.3/100
      Avg Activity Score    83.5/100

✓ Saved: summary_statistics.csv

VISUALIZATION COMPLETE!

📊 Files Created in: /content/drive/MyDrive/vendor-analysis-project/data/exports/visualizations

12 Interactive Visualizations:
  1. Vendor Tier Distribution
  2. Top 15 States by 